In [14]:
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install cryptography PyMySQL[rsa] sqlalchemy pandas

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------------- ---------------- 1.0/1.8 MB 12.0 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 10.9 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ------------------- -------------------- 1.8/3.8 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 11.5 MB/s  0:00:00

   ------------- -------------------------- 1/3 [cffi]
   -------------------------- ------------- 2/3 [cryptography]
   -------------------------- ------------- 2/3 [cryptography]
   -------------------------- ------------- 2/3 [cryptography]
   -------------------------- ------------- 2/3 [cryptography]
   -------------------------- ------------- 2/3 [cryptography]
   ---------------------------------------- 3/3 [cryptography]



In [17]:
# !pip install mysql-connector-python sqlalchemy pandas

from sqlalchemy import create_engine, text

USER = "root"
PASSWORD = "231128101507-{}>=+]["
HOST = "localhost"
PORT = 3306
DATABASE = "sakila"

# Usamos mysql-connector-python en lugar de pymysql
connection_url = f"mysql+mysqlconnector://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"

engine = create_engine(
    connection_url,
    pool_pre_ping=True
)

with engine.connect() as connection:
    result = connection.execute(text("SELECT DATABASE();"))
    print("Conectado con éxito a:", result.fetchone()[0])

Conectado con éxito a: sakila


In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

# --------------------------------------------------
# 1. Establish connection to Sakila
# --------------------------------------------------
USER = "root"
PASSWORD = "231128101507-{}>=+]["
HOST = "localhost"
PORT = 3306
DATABASE = "sakila"

connection_url = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"

engine = create_engine(
    connection_url,
    pool_pre_ping=True
)

with engine.connect() as connection:
    result = connection.execute(text("SELECT DATABASE();"))
    current_db = result.fetchone()[0]
    print("Connected to database:", current_db)


# --------------------------------------------------
# 2. Function: rentals_month
# --------------------------------------------------
def rentals_month(engine, month, year):
    start_date = f"{year}-{month:02d}-01"

    if month == 12:
        end_date = f"{year + 1}-01-01"
    else:
        end_date = f"{year}-{month + 1:02d}-01"

    query = text("""
        SELECT
            rental_id,
            rental_date,
            inventory_id,
            customer_id,
            return_date,
            staff_id
        FROM rental
        WHERE rental_date >= :start_date
          AND rental_date < :end_date
        ORDER BY customer_id, rental_date
    """)

    df = pd.read_sql_query(
        sql=query,
        con=engine,
        params={"start_date": start_date, "end_date": end_date}
    )

    return df


# --------------------------------------------------
# 3. Function: rental_count_month
# --------------------------------------------------
def rental_count_month(df, month, year):
    column_name = f"rentals_{month:02d}_{year}"

    rental_counts = (
        df.groupby("customer_id")
          .size()
          .reset_index(name=column_name)
          .sort_values(by="customer_id")
          .reset_index(drop=True)
    )

    return rental_counts


# --------------------------------------------------
# 4. Function: compare_rentals
# --------------------------------------------------
def compare_rentals(df1, df2):
    comparison = pd.merge(df1, df2, on="customer_id", how="inner")

    col1 = df1.columns[1]
    col2 = df2.columns[1]

    comparison["difference"] = comparison[col2] - comparison[col1]

    return comparison


# --------------------------------------------------
# 5. Run analysis for May and June 2005
# --------------------------------------------------
may_rentals = rentals_month(engine, 5, 2005)
june_rentals = rentals_month(engine, 6, 2005)

may_counts = rental_count_month(may_rentals, 5, 2005)
june_counts = rental_count_month(june_rentals, 6, 2005)

comparison_df = compare_rentals(may_counts, june_counts)


# --------------------------------------------------
# 6. Optional improvement
# --------------------------------------------------
def add_activity_label(df):
    df = df.copy()
    df["activity_change"] = df["difference"].apply(
        lambda x: "increased" if x > 0 else ("decreased" if x < 0 else "same")
    )
    return df


final_report = add_activity_label(comparison_df)


# --------------------------------------------------
# 7. Output
# --------------------------------------------------
print("\n--- Rentals in May 2005 ---")
print(may_rentals.head())

print("\n--- Rentals in June 2005 ---")
print(june_rentals.head())

print("\n--- Rental counts in May 2005 ---")
print(may_counts.head())

print("\n--- Rental counts in June 2005 ---")
print(june_counts.head())

print("\n--- Customers active in both months ---")
print(comparison_df.head())

print("\n--- Final report with activity change ---")
print(final_report.head(10))

Connected to database: sakila

--- Rentals in May 2005 ---
Empty DataFrame
Columns: [rental_id, rental_date, inventory_id, customer_id, return_date, staff_id]
Index: []

--- Rentals in June 2005 ---
Empty DataFrame
Columns: [rental_id, rental_date, inventory_id, customer_id, return_date, staff_id]
Index: []

--- Rental counts in May 2005 ---
Empty DataFrame
Columns: [customer_id, rentals_05_2005]
Index: []

--- Rental counts in June 2005 ---
Empty DataFrame
Columns: [customer_id, rentals_06_2005]
Index: []

--- Customers active in both months ---
Empty DataFrame
Columns: [customer_id, rentals_05_2005, rentals_06_2005, difference]
Index: []

--- Final report with activity change ---
Empty DataFrame
Columns: [customer_id, rentals_05_2005, rentals_06_2005, difference, activity_change]
Index: []
